# TRIDENT T69 ? exact epoch-12 validation benchmark
Read-only inference: **seven tasks x the same 123 validation frames**. No training, fine-tuning, threshold tuning, export or official test samples.

First run all cells with `RUN_BENCHMARK=False`: checkpoint SHA/state/history/source audit and protocol verification must pass before execution.
Inspect the audit. Then set `RUN_BENCHMARK=True` and run the benchmark cell. Each attempt requires a fresh output directory; restart from the top for a new attempt after failure.

Use **Colab ? Runtime ? Change runtime type ? T4 GPU**. Upload the four supplied inputs into the Drive folder below. Keep `epoch_012.pt`, `run.json` and `epoch_012_metrics.json` together in their original completed training directory. Do not replace or edit them.
Allow approximately **8 GB free Drive space** for all predictions, per-frame evidence and panels.

## 1. Mount Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os,sys,json,hashlib,zipfile,subprocess,datetime,uuid
from pathlib import Path,PurePosixPath
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
INPUT=Path('/content/drive/MyDrive/TRIDENT-T69/extended_epoch012_validation_inputs')
SOURCE_ZIP=INPUT/'trident_extended_epoch012_eval_current_source.zip'
VALIDATION_ZIP=INPUT/'nyu_validation123_only.zip'
B0_CHECKPOINT=INPUT/'B0_epoch_002.pt'
TRAINING_LOG=INPUT/'training_console_original.log'
TRAINING_RUN=Path('/content/drive/MyDrive/TRIDENT-T69/runs/extended_v1/20261107T034428Z_aa3a61d441')
CHECKPOINT=TRAINING_RUN/'epoch_012.pt'
RUN_BENCHMARK=False
RUN_ID=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:10]
WORK=Path('/content')/('trident_epoch012_eval_'+RUN_ID)
REPO,DATA=WORK/'repo',WORK/'validation_data'
EVIDENCE=Path('/content/drive/MyDrive/TRIDENT-T69/verification_results/extended_v1_epoch012_preparation')/RUN_ID
OUTPUT=Path('/content/drive/MyDrive/TRIDENT-T69/verification_results/extended_v1_epoch012')/RUN_ID
PY=sys.executable
print('New benchmark output:',OUTPUT)

## 2. Verify Tesla T4, checkpoint SHA256 and exact input packages before deserialization

In [ ]:
import torch
assert torch.cuda.is_available() and torch.cuda.device_count()==1,'Select a CUDA T4 runtime'
assert torch.cuda.get_device_name(0)=='Tesla T4','Use Tesla T4; no silent hardware substitution'
def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
EXPECTED={INPUT/'trident_extended_epoch012_eval_current_source.zip':'c64876f8cb6d15cc0f5784551e03c6a9a9558ca5897807ae0f550def1da56f90', INPUT/'nyu_validation123_only.zip':'9daad64f7641969b8ee515cf5062dcda83339f2d7850fcf2fc6f20104c517267', INPUT/'training_console_original.log':'ee444bba51413d23e0261cfa45f893faea6dd92f505b7c5823789ede228d646c', INPUT/'B0_epoch_002.pt':'a6416c47a16ec78e97c8b734ae7fb23cba4dbbeb5f5564a3e02d61a4a159c1b2'}
EXPECTED[CHECKPOINT]='09bb8fc5edfff40c7fe27a39d44eb571d02726e0bb22a78c28afcd7f0bd844bd'
for p,h in EXPECTED.items():
    assert p.is_file(),f'Missing required input: {p}'
    assert digest(p)==h,f'SHA256 mismatch before model loading: {p}'
    print(p.name,h)
for p in (TRAINING_RUN/'run.json',TRAINING_RUN/'epoch_012_metrics.json'):
    assert p.is_file(),f'Missing original training metadata: {p}'
assert not WORK.exists() and not OUTPUT.exists() and not EVIDENCE.exists()
WORK.mkdir();REPO.mkdir();DATA.mkdir();EVIDENCE.mkdir(parents=True)

## 3. Extract verified current source; preserve original training sources

In [ ]:
def safe_members(z):
    names=z.namelist()
    assert len(names)==len(set(names)),'Duplicate archive member'
    for item in z.infolist():
        p=PurePosixPath(item.filename)
        assert not p.is_absolute() and '..' not in p.parts and '\\' not in item.filename
        assert ((item.external_attr>>16)&0o170000)!=0o120000,'Symlink member forbidden'
    return set(names)
with zipfile.ZipFile(SOURCE_ZIP) as z:
    safe_members(z);z.extractall(REPO)
os.chdir(REPO)

## 4. Install only additional dependencies; preserve CUDA PyTorch and run all tests
The optional original-ZIP B0 unit test may skip because the kit supplies a raw checkpoint. No real-data training is performed by setup.

In [ ]:
# Use Colab's existing runtime directly: no venv or ensurepip.
import tempfile
PY = sys.executable
print("Colab Python:", PY)
torch_probe = """
import json, torch
print(json.dumps(dict(version=str(torch.__version__), cuda=torch.version.cuda,
 location=torch.__file__, cuda_available=torch.cuda.is_available())))
"""
TORCH_BEFORE = json.loads(subprocess.check_output([str(PY), "-c", torch_probe], text=True))
assert TORCH_BEFORE["cuda_available"], "Select a GPU runtime; Colab CUDA PyTorch must already be available"

# Resolve only the additional requirements first. Reject any plan that would
# touch PyTorch or its CUDA stack, then install exactly that plan without resolving
# further dependencies. This cannot pull in a CPU-only torch wheel.
with tempfile.TemporaryDirectory(prefix="trident-pip-") as temporary:
    report = Path(temporary) / "install-plan.json"
    subprocess.run([str(PY), "-m", "pip", "install", "--dry-run",
                    "--only-binary=:all:", "--report", str(report),
                    "-r", str(REPO / "requirements-colab.txt")], check=True)
    plan = json.loads(report.read_text())["install"]
    protected = {"torch", "torchvision", "torchaudio", "torchtext", "triton", "pytorch-triton"}
    for item in plan:
        name = item["metadata"]["name"].lower().replace("_", "-")
        assert name not in protected and not name.startswith("nvidia-"), f"Refusing to replace Colab GPU package: {name}"
    urls = [item["download_info"]["url"] for item in plan]
    if urls:
        subprocess.run([str(PY), "-m", "pip", "install", "--no-deps",
                        "--only-binary=:all:", *urls], check=True)
subprocess.run([str(PY), "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", str(REPO)], check=True)
TORCH_AFTER = json.loads(subprocess.check_output([str(PY), "-c", torch_probe], text=True))
assert TORCH_AFTER == TORCH_BEFORE, "Colab PyTorch/CUDA changed during setup"
probe = """
import json, platform, torch, numpy, scipy, cv2, PIL, h5py
assert (2, 6) <= tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (3, 0)
print(json.dumps(dict(python=platform.python_version(), torch=str(torch.__version__),
 cuda=torch.version.cuda, numpy=numpy.__version__, scipy=scipy.__version__,
 opencv=cv2.__version__, pillow=PIL.__version__, h5py=h5py.__version__,
 gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
 gpu_memory_bytes=torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None)))
"""
ENVIRONMENT = json.loads(subprocess.check_output([str(PY), "-c", probe], text=True))
print(json.dumps(ENVIRONMENT, indent=2))
assert ENVIRONMENT["gpu"], "No CUDA GPU available: select a GPU runtime. Do not run the pilot on CPU."
# Complete project suite after installation; CPU tests do not train a model.
subprocess.run([str(PY), "-m", "pytest", "-q"], check=True,
               env=dict(os.environ, CUDA_VISIBLE_DEVICES=""))

## 5. Inspect exact epoch-12 checkpoint, original logs and per-frame training metadata
This audit hashes before loading. It rejects missing or inconsistent original files, history, source, configuration, optimizer or scheduler state. Do not bypass a failed audit.

In [ ]:
AUDIT=EVIDENCE/'checkpoint_audit.json'
command=[PY,'-B','-m','trident_depth.extended_v1.evaluation_audit',
         '--checkpoint',str(CHECKPOINT),'--training-log',str(TRAINING_LOG),
         '--config',str(REPO/'configs/extended_v1/train.json'),
         '--reference',str(REPO/'docs/pilot_dense4/per_frame.json'),'--output',str(AUDIT)]
subprocess.run(command,cwd=REPO,check=True)
checkpoint_audit=json.loads(AUDIT.read_text())
assert checkpoint_audit['status']=='PASS' and checkpoint_audit['best_epoch']==12
print(json.dumps(checkpoint_audit,indent=2))
print('These are RECORDED training-validation metrics, not new benchmark results.')

## 6. Stage only validation payloads and verify the frozen evaluation protocol

In [ ]:
with zipfile.ZipFile(VALIDATION_ZIP) as z:
    names=safe_members(z)
    raw=z.read('manifest_development.json')
    assert hashlib.sha256(raw).hexdigest()=='d1cdbcf0ff3ee96f2e0c173c3c4701ed3f5c526ab517ce3b988268c3c6a391f5'
    rows=[r for r in json.loads(raw) if r['split']=='validation']
    allowed={'manifest_development.json'}|{r[k] for r in rows for k in ('rgb','depth','mask')}
    assert len(rows)==123 and len(names)==370 and names==allowed,'Validation-only payloads required'
    z.extractall(DATA)
PROTOCOL=REPO/'configs/extended_v1/validation_epoch012.json'
assert digest(PROTOCOL)=='f147821bfe900263ac022b05df0c679e745978d5e6af5b4a66ec8f2ae09fae67'
print(PROTOCOL.read_text())
# Run a separate, guarded payload and scene audit in a clean process.
code_to_run="""
import json,sys
from pathlib import Path
from trident_depth.evaluation.hybrid_v1_data import ValidationOnly
from trident_depth.extended_v1.data import pipeline
from trident_depth.extended_v1.contract import read_config,environment_identity
from trident_depth.extended_v1.checkpoint import load
root,manifest,out,checkpoint=map(Path,sys.argv[1:])
cfg=read_config(root/'configs/extended_v1/train.json')
_,payload=load(checkpoint)
assert environment_identity()==payload['provenance']['environment'], 'Original training environment differs; stop, do not substitute'
val=ValidationOnly(manifest,root/'docs/pilot_dense4/per_frame.json')
with val.isolated():
    val.prepare(out,pipeline(cfg['tasks'][1]))
    assert not val.denied
report=dict(status='PASS',validation_frames=123,scenes=30,validation_payload_files=369,
            official_test_sample_files_opened=0,training_sample_files_opened=0,
            environment_matches_original=True,protocol_frame_pairs=861)
with (out/'dataset_protocol_audit.json').open('x') as f:json.dump(report,f,indent=2)
print(json.dumps(report,indent=2))
"""
subprocess.run([PY,'-B','-c',code_to_run,str(REPO),str(DATA/'manifest_development.json'),str(EVIDENCE),str(CHECKPOINT)],cwd=REPO,check=True)
with (EVIDENCE/'pip_freeze.txt').open('x') as f:subprocess.run([PY,'-m','pip','freeze'],stdout=f,check=True)
with (EVIDENCE/'input_provenance.json').open('x') as f:json.dump({str(p):h for p,h in EXPECTED.items()},f,indent=2)
print('Preparation passed. Review checkpoint_audit.json and dataset_protocol_audit.json before execution.')

## 7. Run the validation benchmark only after reviewing the audits
Set `RUN_BENCHMARK=True` in this cell when ready. Baselines: nearest/bilinear/bicubic with filling, joint bilateral, guided and classical; B0 only on dense4; raw extended and existing extended hybrid. No thresholds can be tuned through notebook arguments. Each method has one warm-up plus three timed repetitions. All 861 task/frame pairs are retained.

In [ ]:
# Change this to True after reviewing the successful audits above.
RUN_BENCHMARK=False
if not RUN_BENCHMARK:
    print('Benchmark not executed. Audits are saved at',EVIDENCE)
else:
    assert json.loads(AUDIT.read_text())['status']=='PASS'
    assert json.loads((EVIDENCE/'dataset_protocol_audit.json').read_text())['status']=='PASS'
    for p,h in EXPECTED.items():assert digest(p)==h,f'Original input changed: {p}'
    assert not OUTPUT.exists(),'Fresh output required; never overwrite an earlier run'
    command=[PY,'-B','-m','trident_depth.extended_v1.evaluation','--execute-validation',
             '--checkpoint',str(CHECKPOINT),'--training-log',str(TRAINING_LOG),
             '--manifest',str(DATA/'manifest_development.json'),'--b0-checkpoint',str(B0_CHECKPOINT),
             '--config',str(REPO/'configs/extended_v1/train.json'),'--protocol',str(PROTOCOL),
             '--reference',str(REPO/'docs/pilot_dense4/per_frame.json'),'--output',str(OUTPUT)]
    with (EVIDENCE/'benchmark_launch.json').open('x') as f:json.dump(dict(command=command),f,indent=2)
    with (EVIDENCE/'benchmark_console.log').open('x') as log:
        process=subprocess.Popen(command,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='');log.write(line);log.flush()
        returncode=process.wait()
    assert returncode==0,f'Benchmark failed. Preserve {OUTPUT} and {EVIDENCE}. Stop; do not tune or overwrite evidence.'

## 8. Review actual metrics, failure evidence and isolation audit
No additional execution follows this cell. Compare hole-interior performance, not only whole-image SSIM or improvements from retaining observed points.

In [ ]:
if RUN_BENCHMARK:
    report=json.loads((OUTPUT/'report.json').read_text())
    assert report['status']=='COMPLETE','A replay failure is not a valid completed comparison'
    assert report['unique_validation_frames']==123 and report['task_frame_pairs']==861
    assert report['isolation']['official_test_sample_files_opened']==0
    assert report['isolation']['training_sample_files_opened']==0
    assert not report['isolation']['forbidden_access_attempts']
    for p,h in EXPECTED.items():assert digest(p)==h,f'Input changed: {p}'
    from IPython.display import display,Markdown,Image
    display(Markdown((OUTPUT/'report.md').read_text()))
    for task in ('incomplete','sparse_noisy_holes'):
        worst=report['panels'][task]['worst_extended_rmse_ids'][0]
        display(Image(filename=str(OUTPUT/'panels'/f'{task}_{worst}.png')))
    print('Complete reports, predictions, panels, latency/memory and access audit:',OUTPUT)
else:
    print('No benchmark results yet. Complete the audit review, then enable the execution cell.')